# Flatland TorchRL — Colab/Kaggle Notebook
Run `flatland_ppo_training_torchrl.py` and `scripts/puffer_sweep.py` with GPU.

**Notes**:
- This notebook assumes CUDA is available.
- It clones the full repo so local modules and configs resolve.
- For private repos, set `GITHUB_TOKEN` and (optionally) `FLATLAND_REPO` in the notebook environment.
- Logs go to `runs/` and sweep observations to `puffer_runs/`.


In [1]:
# Clone repo
!git clone https://github.com/louiesmrs/flatland-torchrl.git

fatal: destination path 'flatland-torchrl' already exists and is not an empty directory.


In [2]:
# Remove any local virtualenv that can shadow system packages
!rm -rf .venv
!find . -name pyvenv.cfg -delete

In [3]:
%cd /content/flatland-torchrl

/content/flatland-torchrl


In [4]:
# System deps (needed for flatland_cutils build on Colab)
!apt-get update -y
!apt-get install -y build-essential

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cli.github.com/packages stable InRelease                         
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease               
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease                 
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease          
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease    
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/u

In [5]:
# Install Python deps (CUDA)
!pip install -U pip
!pip install uv
!uv pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Using Python 3.12.12 environment at: /usr
Resolved 27 packages in 3.23s                                        
Prepared 14 packages in 52.16s                                           
Uninstalled 14 packages in 340ms
Installed 14 packages in 329ms                              
 - nvidia-cublas-cu12==12.8.4.1
 + nvidia-cublas-cu12==12.1.3.1
 - nvidia-cuda-cupti-cu12==12.8.90
 + nvidia-cuda-cupti-cu12==12.1.105
 - nvidia-cuda-nvrtc-cu12==12.8.93
 + nvidia-cuda-nvrtc-cu12==12.1.105
 - nvidia-cuda-runtime-cu12==12.8.90
 + nvidia-cuda-runtime-cu12==12.1.105
 - nvidia-cudnn-cu12==9.10.2.21
 + nvidia-cudnn-cu12==9.1.0.70
 - nvidia-cufft-cu12==11.3.3.83
 + nvidia-cufft-cu12==11.0.2.54
 - nvidia-curand-cu12==10.3.9.90
 + nvidia-curand-cu12==10.3.2.106
 - nvidia-cusolver-cu12==11.7.3.90
 + nvidia-cusolver-cu12==11.4.5.107
 - nvidia-cusparse-cu12==12.5.8.93
 + nvidia-cusparse-cu12==12.1.0.106
 - nvidia-nccl-cu12==2.27.5
 + nvidia-nccl-cu12==2.21.5
 - nvidia-nvtx-cu12==12.8.90
 + nvidia-nvtx-cu1

In [6]:
# Install repo + flatland_cutils
!uv pip  install -e .

Using Python 3.12.12 environment at: /usr
Resolved 224 packages in 959ms                                       
Prepared 1 package in 777ms                                              
Uninstalled 15 packages in 168ms
Installed 15 packages in 224ms                              
 ~ flatland-marl-ppo==0.1.0 (from file:///content/flatland-torchrl)
 - nvidia-cublas-cu12==12.1.3.1
 + nvidia-cublas-cu12==12.8.4.1
 - nvidia-cuda-cupti-cu12==12.1.105
 + nvidia-cuda-cupti-cu12==12.8.90
 - nvidia-cuda-nvrtc-cu12==12.1.105
 + nvidia-cuda-nvrtc-cu12==12.8.93
 - nvidia-cuda-runtime-cu12==12.1.105
 + nvidia-cuda-runtime-cu12==12.8.90
 - nvidia-cudnn-cu12==9.1.0.70
 + nvidia-cudnn-cu12==9.10.2.21
 - nvidia-cufft-cu12==11.0.2.54
 + nvidia-cufft-cu12==11.3.3.83
 - nvidia-curand-cu12==10.3.2.106
 + nvidia-curand-cu12==10.3.9.90
 - nvidia-cusolver-cu12==11.4.5.107
 + nvidia-cusolver-cu12==11.7.3.90
 - nvidia-cusparse-cu12==12.1.0.106
 + nvidia-cusparse-cu12==12.5.8.93
 - nvidia-nccl-cu12==2.21.5
 + nvid

In [7]:
# Use non-editable install for flatland_cutils to avoid import shadowing in notebooks
!uv pip  install ./flatland_cutils

Using Python 3.12.12 environment at: /usr
Resolved 1 package in 573ms                                          
Prepared 1 package in 300ms                                              
Uninstalled 1 package in 1ms
Installed 1 package in 1ms0.1 (from file:///content/flatland
 ~ flatland-cutils==0.0.1 (from file:///content/flatland-torchrl/flatland_cutils)


In [8]:
!uv run python -c "import sys,runpy; \
sys.path.insert(0,'/content/flatland-torchrl'); \
sys.argv=['flatland_ppo_training_torchrl.py','--pretrained-network-path','model_checkpoints/flatland-rl__jiang_phase_1_3_7_to_10_agents__1__1769903932/flatland-rl__jiang_phase_1_3_7_to_10_agents__1__1769903932_4480000.tar','--num-envs','10','--num-steps','200','--vf-coef','0.13','--ent-coef','0.001','--max-grad-norm','0.2','--learning-rate','4.07e-5','--clip-coef','0.29','--seed','1','--exp-name','jiang_phase_1_3_7_to_10_agents','--curriculum-path','curriculums/jiang_phases_1_3_7_to_10_agents_30x30.json','--value-loss','l2']; \
runpy.run_path('/content/flatland-torchrl/flatland_ppo_training_torchrl.py', run_name='main')"

: 

In [ ]:
!uv pip install -r tools/requirements.txt

Using Python 3.12.12 environment at: /usr
⠦ Resolving dependencies...                                                     

In [9]:
# Sweep run (force system site-packages before repo path)
!python -c "import sys,runpy; \
sys.path.insert(0,'/content/flatland-torchrl'); \
sys.argv=['puffer_sweep.py','--config','scripts/flatland_sweep.yaml','--use-gpu']; \
runpy.run_path('/content/flatland-torchrl/scripts/puffer_sweep.py', run_name='__main__')"

Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "<frozen runpy>", line 287, in run_path
  File "<frozen runpy>", line 98, in _run_module_code
  File "<frozen runpy>", line 88, in _run_code
  File "/content/flatland-torchrl/scripts/puffer_sweep.py", line 14, in <module>
    from pufferlib.sweep import Protein
ModuleNotFoundError: No module named 'pufferlib'


## TensorBoard
In Colab, run:
```
%load_ext tensorboard
%tensorboard --logdir runs
```

In [10]:
# Bundle artifacts
!tar -czf artifacts.tgz runs puffer_runs

# Colab download helper
try:
    from google.colab import files

    files.download("artifacts.tgz")
except Exception:
    print("If running on Kaggle, find artifacts.tgz in /kaggle/working")

tar: puffer_runs: Cannot stat: No such file or directory
tar: Exiting with failure status due to previous errors


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>